# Track 2 — Sentinel-2 segmentation: per-pixel RF vs U-Net (cloud GPU)

A thin wrapper around [`scripts/train_segmentation.py`](https://github.com/tarpous/landcover-sentinel2/blob/main/scripts/train_segmentation.py). Fetches a real Sentinel-2 median composite + ESA WorldCover labels over an AOI, then compares a per-pixel Random Forest against a U-Net with a label-efficiency curve. Results land in `results/segmentation.json`.

In [ ]:
# Thin wrapper: clone the repo and install the training extras. The real logic
# lives in the tested `landcover` package and `scripts/train_*.py`; this notebook
# only runs them on a cloud GPU for anyone without a local one.
import subprocess, sys, os
if not os.path.exists("landcover-sentinel2"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/tarpous/landcover-sentinel2.git"], check=True)
%cd landcover-sentinel2
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "timm", "segmentation-models-pytorch"], check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'pystac-client', 'stackstac'], check=True)

In [ ]:
subprocess.run([sys.executable, 'scripts/fetch_data.py',
                '--aoi', '22.90', '40.55', '23.02', '40.64', '--chip', '256'], check=True)

In [ ]:
subprocess.run([sys.executable, 'scripts/train_segmentation.py',
                '--root', 'data/raw/chips', '--epochs', '60', '--device', 'cuda',
                '--block-size', '512'], check=True)
subprocess.run([sys.executable, 'scripts/make_results_table.py'], check=True)